# MatGPT Telco 300M — guarded Colab training

This notebook builds a 306,226,176-parameter English and telecom base model. Run one stage at a time. Data, evidence, and checkpoints persist in Google Drive; temporary tokenizer and shard copies stay on the local runtime for speed.

## 1. Choose one stage

In [ ]:
RUN_STAGE = "prepare_data"  # @param ["prepare_data", "prepare", "smoke", "pilot", "full", "evaluate"]
DATA_PLAN = "pilot"  # @param ["pilot", "full"]
ALLOW_FULL_DATA = False  # @param {type:"boolean"}
FULL_APPROVED = False  # @param {type:"boolean"}
PILOT_TOKENS = 20_000_000
SMOKE_MAX_STEPS = 20
SMOKE_RESUME_STEPS = 5
PILOT_MAX_ADDITIONAL_STEPS = 200
STAGES = {"prepare_data", "prepare", "smoke", "pilot", "full", "evaluate"}
assert RUN_STAGE in STAGES
assert DATA_PLAN in {"pilot", "full"}
if RUN_STAGE in {"smoke", "pilot"}:
    assert DATA_PLAN == "pilot", f"{RUN_STAGE} requires DATA_PLAN='pilot'."
if RUN_STAGE == "full":
    assert DATA_PLAN == "full", "Full training requires DATA_PLAN='full'."
print(f"Selected stage={RUN_STAGE!r}, data plan={DATA_PLAN!r}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Locate or clone the project

In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/train-llm-from-scratch")
REPOSITORY = "https://github.com/digotetso/train-llm-from-scratch.git"
if not (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(PROJECT_DIR)
print(subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())

## 4. Install and authenticate

In [ ]:
import getpass
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
from huggingface_hub import login
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face read token (input is hidden): " ).strip()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    print("No token supplied. Public sources may still work, subject to Hugging Face access rules.")

## 5. Inspect the runtime

In [ ]:
import shutil
import torch

print("/content disk:", shutil.disk_usage("/content"))
print("Drive disk:", shutil.disk_usage("/content/drive"))
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if RUN_STAGE in {"prepare", "smoke", "pilot", "full", "evaluate"}:
    assert torch.cuda.is_available(), f"{RUN_STAGE} requires CUDA."
    properties = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    print("gpu:", gpu_name, "memory GiB:", properties.total_memory / 1024**3)
    if properties.total_memory < 40 * 1024**3:
        print("Warning: less than 40 GiB VRAM; reduce micro-batch size and re-benchmark.")

## 6. Build fixed local and Drive paths

In [ ]:
import json
import yaml

WORK_ROOT = Path("/content/matgpt_work") / "matgpt_telco_300m"
DRIVE_ROOT = Path("/content/drive/MyDrive/matgpt_artifacts") / "matgpt_telco_300m"
CHECKED_CONFIG = PROJECT_DIR / "configs/matgpt_telco_300m.yaml"
SOURCE_REGISTRY = PROJECT_DIR / "configs/data/telco_300m_sources.yaml"
MIXTURE_CONFIG = PROJECT_DIR / "configs/data/telco_300m_mixture.yaml"
CORPUS_DIR = DRIVE_ROOT / "corpora" / DATA_PLAN
EVAL_LITE_DIR = DRIVE_ROOT / "evaluation_assets/open_telco_lite"
EVAL_FULL_DIR = DRIVE_ROOT / "evaluation_assets/open_telco_full"
TOKENIZER_DIR = WORK_ROOT / DATA_PLAN / "tokenizer"
SHARD_DIR = WORK_ROOT / DATA_PLAN / "shards"
ARTIFACT_DRIVE_DIR = DRIVE_ROOT / "prepared" / DATA_PLAN
PILOT_TOKENIZER_DRIVE_DIR = DRIVE_ROOT / "prepared/pilot/tokenizer"
EVIDENCE_DIR = DRIVE_ROOT / "evidence" / DATA_PLAN
RUN_DIR = DRIVE_ROOT / "runs" / DATA_PLAN
CONFIG_PATH = WORK_ROOT / DATA_PLAN / "config/matgpt_telco_300m.yaml"
for path in (WORK_ROOT, DRIVE_ROOT, EVIDENCE_DIR, RUN_DIR, CONFIG_PATH.parent):
    path.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(CHECKED_CONFIG.read_text(encoding="utf-8"))
cfg["dataset"]["source_registry_path"] = str(SOURCE_REGISTRY)
cfg["dataset"]["mixture_config_path"] = str(MIXTURE_CONFIG)
cfg["dataset"]["normalized_dir"] = str(CORPUS_DIR)
cfg["tokenizer"]["output_dir"] = str(TOKENIZER_DIR)
cfg["tokenizer"]["probe_sets_path"] = str(PROJECT_DIR / "configs/data/telco_tokenizer_probes.yaml")
cfg["sharding"]["output_dir"] = str(SHARD_DIR)
cfg["run"]["output_dir"] = str(RUN_DIR)
if DATA_PLAN == "pilot":
    cfg["dataset"]["train_split"] = "pilot"
    cfg["dataset"]["training_splits"] = {"pilot": "pilot"}
    cfg["training"]["max_tokens"] = PILOT_TOKENS
    cfg["training"]["data_phases"] = [{"name": "pilot", "split": "pilot", "until_tokens": PILOT_TOKENS}]
    cfg["training"]["eval_interval_tokens"] = 5_000_000
    cfg["training"]["checkpoint_interval_tokens"] = 5_000_000
    cfg["training"]["sample_interval_tokens"] = 5_000_000
CONFIG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
from matgpt.config import config_to_yaml, load_config
from matgpt.utils.hashing import sha256_file, sha256_text
CURRENT_CONFIG_SHA = sha256_text(config_to_yaml(load_config(CONFIG_PATH)))
print("Config:", CONFIG_PATH, "Corpus:", CORPUS_DIR, "Run:", RUN_DIR)

## 7. Prepare isolated evaluation and training data

In [ ]:
def run_command(command):
    print("$", " ".join(map(str, command)))
    result = subprocess.run(list(map(str, command)), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, command))}")
    return result

from matgpt.data.telco_prepare import corpus_has_exact_token_quotas

def build_corpus_command(stages, *, tokenizer_dir=None, force=False):
    command = [sys.executable, "scripts/prepare_telco_corpus.py", "--sources", SOURCE_REGISTRY, "--mixture", MIXTURE_CONFIG]
    for stage in stages:
        command += ["--stage", stage]
    command += ["--output-dir", CORPUS_DIR]
    for pattern_path in sorted(EVAL_LITE_DIR.glob("*.jsonl")) + sorted(EVAL_FULL_DIR.glob("*.jsonl")):
        command += ["--contamination-patterns", pattern_path]
    if DATA_PLAN == "pilot":
        command += ["--total-tokens", str(PILOT_TOKENS)]
    else:
        command += ["--allow-full-data"]
    if tokenizer_dir is not None:
        command += ["--tokenizer-dir", tokenizer_dir]
    if force:
        command += ["--force"]
    return command

if RUN_STAGE == "prepare_data":
    for dataset_name, destination in (("lite", EVAL_LITE_DIR), ("full", EVAL_FULL_DIR)):
        if not (destination / "manifest.json").is_file():
            run_command([sys.executable, "scripts/prepare_open_telco_evals.py", "--sources", SOURCE_REGISTRY, "--dataset", dataset_name, "--output-dir", destination])
    stages = ["pilot"] if DATA_PLAN == "pilot" else ["main", "cooldown"]
    if DATA_PLAN == "full":
        assert ALLOW_FULL_DATA, "Set ALLOW_FULL_DATA=True to authorize the 12B-token corpus build."
        assert (PILOT_TOKENIZER_DRIVE_DIR / "tokenizer.json").is_file() and (PILOT_TOKENIZER_DRIVE_DIR / "special_tokens.json").is_file(), "Run the pilot prepare stage first; full data must use its frozen tokenizer."
        drive_free = shutil.disk_usage("/content/drive").free
        assert drive_free >= 100 * 1024**3, f"Full preparation needs at least 100 GiB free in Drive; observed {drive_free / 1024**3:.1f} GiB."
    plan_paths = []
    for stage in stages:
        plan_path = EVIDENCE_DIR / f"mixture_plan_{stage}.json"
        command = [sys.executable, "scripts/plan_telco_mixture.py", "--sources", SOURCE_REGISTRY, "--mixture", MIXTURE_CONFIG, "--stage", stage, "--output", plan_path]
        if stage == "pilot":
            command += ["--total-tokens", str(PILOT_TOKENS)]
        run_command(command)
        plan_paths.append(plan_path)
    plans = [json.loads(path.read_text(encoding="utf-8")) for path in plan_paths]
    corpus_manifest_exists = (CORPUS_DIR / "manifest.json").is_file()
    corpus_ready = corpus_manifest_exists
    if DATA_PLAN == "full":
        corpus_ready = corpus_has_exact_token_quotas(CORPUS_DIR, PILOT_TOKENIZER_DRIVE_DIR, plans)
    if not corpus_ready:
        tokenizer_for_quota = PILOT_TOKENIZER_DRIVE_DIR if DATA_PLAN == "full" else None
        run_command(build_corpus_command(stages, tokenizer_dir=tokenizer_for_quota, force=corpus_manifest_exists))
    print("Data preparation complete. Next choose RUN_STAGE='prepare'.")
else:
    print("Skipped: this cell acts only when RUN_STAGE='prepare_data'.")

## 8. Prepare tokenizer and shards

In [ ]:
def atomic_snapshot(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.with_name(destination.name + ".staging")
    backup = destination.with_name(destination.name + ".previous")
    if staging.exists():
        shutil.rmtree(staging)
    shutil.copytree(source, staging)
    if backup.exists():
        shutil.rmtree(backup)
    if destination.exists():
        destination.replace(backup)
    staging.replace(destination)

def tokenizer_artifact_sha256(path):
    try:
        metadata = json.loads((path / "special_tokens.json").read_text(encoding="utf-8"))
        actual = sha256_file(path / "tokenizer.json")
    except (OSError, KeyError, json.JSONDecodeError):
        return None
    return actual if metadata.get("tokenizer_sha256") == actual else None

if RUN_STAGE == "prepare":
    assert (CORPUS_DIR / "manifest.json").is_file(), "Run prepare_data first."
    if DATA_PLAN == "full":
        local_free = shutil.disk_usage("/content").free
        assert local_free >= 35 * 1024**3, f"Full preparation needs at least 35 GiB free under /content; observed {local_free / 1024**3:.1f} GiB."
    stages = ["pilot"] if DATA_PLAN == "pilot" else ["main", "cooldown"]
    plan_paths = [EVIDENCE_DIR / f"mixture_plan_{stage}.json" for stage in stages]
    plans = [json.loads(path.read_text(encoding="utf-8")) for path in plan_paths]
    frozen_tokenizer_dir = PILOT_TOKENIZER_DRIVE_DIR
    frozen_tokenizer_sha256 = tokenizer_artifact_sha256(frozen_tokenizer_dir)
    local_tokenizer_sha256 = tokenizer_artifact_sha256(TOKENIZER_DIR)
    if frozen_tokenizer_sha256 is not None:
        if local_tokenizer_sha256 != frozen_tokenizer_sha256:
            atomic_snapshot(frozen_tokenizer_dir, TOKENIZER_DIR)
    elif DATA_PLAN == "full":
        raise AssertionError("Run the pilot prepare stage first; its frozen tokenizer is required for full preparation.")
    else:
        if local_tokenizer_sha256 is None:
            run_command([sys.executable, "scripts/train_tokenizer.py", "--config", CONFIG_PATH])
            local_tokenizer_sha256 = tokenizer_artifact_sha256(TOKENIZER_DIR)
            assert local_tokenizer_sha256 is not None, "Tokenizer training produced invalid artifacts."
        atomic_snapshot(TOKENIZER_DIR, frozen_tokenizer_dir)
        frozen_tokenizer_sha256 = local_tokenizer_sha256
    print("Frozen tokenizer:", frozen_tokenizer_sha256)
    if not corpus_has_exact_token_quotas(CORPUS_DIR, TOKENIZER_DIR, plans):
        run_command(build_corpus_command(stages, tokenizer_dir=TOKENIZER_DIR, force=True))
    assert corpus_has_exact_token_quotas(CORPUS_DIR, TOKENIZER_DIR, plans), "Corpus is not bound to the frozen tokenizer and current plans."
    audit_command = [sys.executable, "scripts/audit_telco_corpus.py", "--tokenizer-dir", TOKENIZER_DIR, "--tolerance", "0.03", "--output", EVIDENCE_DIR / "quota_audit.json"]
    for stage in stages:
        audit_command += ["--input", CORPUS_DIR / f"{stage}.jsonl", "--plan", EVIDENCE_DIR / f"mixture_plan_{stage}.json"]
    run_command(audit_command)
    run_command([sys.executable, "scripts/tokenize_and_shard.py", "--config", CONFIG_PATH])
    run_command([sys.executable, "scripts/preflight_t4.py", "--config", CONFIG_PATH, "--report-path", EVIDENCE_DIR / "preflight.json", "--require-supported-gpu", "--min-free-disk-gb", "10"] )
    benchmark = run_command([sys.executable, "scripts/benchmark_t4.py", "--config", CONFIG_PATH, "--batch-sizes", "4,8,12", "--steps", "3"] )
    benchmark_payload = json.loads(benchmark.stdout)
    benchmark_payload["config_sha256"] = CURRENT_CONFIG_SHA
    configured_batch = next(row for row in benchmark_payload["results"] if row["batch_size"] == 8)
    assert configured_batch["status"] == "ok", f"Configured batch 8 failed: {configured_batch}"
    assert configured_batch["memory_fraction"] <= 0.90, f"Configured batch 8 has insufficient memory headroom: {configured_batch}"
    (EVIDENCE_DIR / "benchmark.json").write_text(json.dumps(benchmark_payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    if DATA_PLAN == "full":
        atomic_snapshot(TOKENIZER_DIR, ARTIFACT_DRIVE_DIR / "tokenizer")
    atomic_snapshot(SHARD_DIR, ARTIFACT_DRIVE_DIR / "shards")
    shutil.copy2(CONFIG_PATH, ARTIFACT_DRIVE_DIR / "config.yaml")
    print("Artifact preparation complete. Next choose RUN_STAGE='smoke'.")
else:
    print("Skipped: this cell acts only when RUN_STAGE='prepare'.")

## 9. Verify evidence gates

In [ ]:
def restore_prepared_artifacts():
    for name, local in (("tokenizer", TOKENIZER_DIR), ("shards", SHARD_DIR)):
        saved = ARTIFACT_DRIVE_DIR / name
        assert saved.is_dir(), f"Missing prepared {name}: {saved}"
        if not local.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(saved, local)

if RUN_STAGE in {"smoke", "pilot", "full", "evaluate"}:
    for evidence_name in ("quota_audit.json", "preflight.json", "benchmark.json"):
        assert (EVIDENCE_DIR / evidence_name).is_file(), f"Missing evidence: {evidence_name}"
    assert json.loads((EVIDENCE_DIR / "quota_audit.json").read_text())["passed"] is True
    prepared_preflight = json.loads((EVIDENCE_DIR / "preflight.json").read_text())
    assert prepared_preflight["status"] == "pass"
    prepared_config = next(check for check in prepared_preflight["checks"] if check["name"] == "config")
    assert prepared_config["details"]["config_sha256"] == CURRENT_CONFIG_SHA, "Prepared preflight belongs to a different config."
    assert json.loads((EVIDENCE_DIR / "benchmark.json").read_text())["config_sha256"] == CURRENT_CONFIG_SHA, "Benchmark belongs to a different config."
    restore_prepared_artifacts()
    run_command([sys.executable, "scripts/preflight_t4.py", "--config", CONFIG_PATH, "--report-path", EVIDENCE_DIR / f"preflight_{RUN_STAGE}.json", "--require-supported-gpu", "--min-free-disk-gb", "10"] )
if RUN_STAGE == "pilot":
    assert (EVIDENCE_DIR / "smoke_resume_verified.json").is_file(), "Run smoke first."
if RUN_STAGE == "full":
    assert FULL_APPROVED, "Set FULL_APPROVED=True only after reviewing pilot evidence."
    pilot_gate = DRIVE_ROOT / "evidence/pilot/pilot_complete.json"
    assert pilot_gate.is_file(), f"Missing completed pilot gate: {pilot_gate}"
print("Evidence gates checked for", RUN_STAGE)

## 10. Run the selected stage

In [ ]:
SMOKE_MAX_STEPS = 20
SMOKE_RESUME_STEPS = 5
latest = RUN_DIR / "checkpoints/latest.pt"
if RUN_STAGE == "smoke":
    command = [sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--max-steps", str(SMOKE_MAX_STEPS)]
    if latest.is_file():
        command += ["--resume-from", latest]
    run_command(command)
    run_command([sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--resume-from", latest, "--verify-only"] )
    run_command([sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--resume-from", latest, "--max-steps", str(SMOKE_RESUME_STEPS)] )
    torch.save(torch.load(latest, map_location="cpu", weights_only=False)["state"], EVIDENCE_DIR / "smoke_resume_state.pt")
    (EVIDENCE_DIR / "smoke_resume_verified.json").write_text(json.dumps({"status": "pass", "checkpoint": str(latest)}, indent=2) + "\n")
elif RUN_STAGE == "pilot":
    assert latest.is_file(), "Smoke checkpoint is missing."
    run_command([sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH, "--resume-from", latest, "--max-steps", str(PILOT_MAX_ADDITIONAL_STEPS)] )
    state = torch.load(latest, map_location="cpu", weights_only=False)["state"]
    assert int(state["tokens_processed"]) >= PILOT_TOKENS, "Pilot did not reach its token target."
    (EVIDENCE_DIR / "pilot_complete.json").write_text(json.dumps({"status": "pass", "tokens_processed": int(state["tokens_processed"]), "checkpoint": str(latest)}, indent=2) + "\n")
elif RUN_STAGE == "full":
    assert FULL_APPROVED
    command = [sys.executable, "scripts/pretrain.py", "--config", CONFIG_PATH]
    if latest.is_file():
        command += ["--resume-from", latest]
    run_command(command)
else:
    print("No training started for this stage.")

## 11. Evaluate checkpoints

In [ ]:
if RUN_STAGE == "evaluate":
    from datetime import datetime, timezone
    checkpoint_dir = RUN_DIR / "checkpoints"
    candidates = [checkpoint_dir / "best.pt"] + sorted(checkpoint_dir.glob("ckpt_*.pt")) + [checkpoint_dir / "latest.pt"]
    checkpoints = []
    for checkpoint in candidates:
        if checkpoint.is_file() and checkpoint not in checkpoints:
            checkpoints.append(checkpoint)
    assert checkpoints, f"No checkpoints found in {checkpoint_dir}"
    evaluation_dir = RUN_DIR / "evaluation" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    evaluation_dir.mkdir(parents=True, exist_ok=False)
    task_dir = EVAL_FULL_DIR if (EVAL_FULL_DIR / "manifest.json").is_file() else EVAL_LITE_DIR
    tasks = sorted(task_dir.glob("*.jsonl"))
    for index, checkpoint in enumerate(checkpoints):
        label = f"checkpoint_{index:02d}_{checkpoint.stem}"
        run_command([sys.executable, "scripts/evaluate.py", "--config", CONFIG_PATH, "--checkpoint", checkpoint, "--output", evaluation_dir / f"{label}_base.json"] )
        task_command = [sys.executable, "scripts/evaluate_tasks.py", "--config", CONFIG_PATH, "--checkpoint", checkpoint, "--output", evaluation_dir / f"{label}_open_telco.json"]
        for task in tasks:
            task_command += ["--task", task]
        run_command(task_command)
    if len(checkpoints) >= 2:
        comparison_dir = evaluation_dir / "checkpoint_comparison"
        command = [sys.executable, "scripts/compare_checkpoints.py", "--config", CONFIG_PATH, "--review-per-checkpoint", "50", "--output-dir", comparison_dir]
        for index, checkpoint in enumerate(checkpoints):
            command += ["--checkpoint", f"checkpoint_{index:02d}={checkpoint}"]
        run_command(command)
        llm_judge = comparison_dir / "llm_judge"
        print(f"LLM judge bundle: {llm_judge}")
        print("Attach judge_prompt.md and each blinded batch from llm_judge/batches to this Codex task. Ask Codex to return judgment JSONL, then run scripts/score_story_judgments.py. Human review is optional.")
    else:
        print("One checkpoint evaluated. Preserve at least two checkpoints to create a blinded comparison bundle.")
else:
    print("Skipped: this cell acts only when RUN_STAGE='evaluate'.")

## 12. Review persisted evidence

In [ ]:
print("Drive root:", DRIVE_ROOT)
for path in sorted(EVIDENCE_DIR.glob("*")):
    print("evidence:", path.name, path.stat().st_size, "bytes")
if (RUN_DIR / "metrics.csv").is_file():
    print("metrics:", RUN_DIR / "metrics.csv")
if (RUN_DIR / "checkpoints/latest.pt").is_file():
    state = torch.load(RUN_DIR / "checkpoints/latest.pt", map_location="cpu", weights_only=False)["state"]
    print("latest checkpoint state:", {key: state.get(key) for key in ("global_step", "tokens_processed", "best_val_loss")})